# Analyze the LeRobot v3 image loading regression

This notebook reads the two TraceML SQLite databases produced by `run_reproduction.sh`. It validates timing coverage, shows TraceML's comparison, and plots per-step DataLoader wait and total step time.

Run it on the machine where you copied the result bundle. No GPU or TraceML installation is required.

## Local setup

From the TraceML repository root:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r examples/advanced/lerobot_v3_image_regression/analysis-requirements.txt
jupyter lab examples/advanced/lerobot_v3_image_regression/analyze_results.ipynb
```

Then choose **Run All**. The newest complete local result is selected automatically. Set `RESULT_DIR` below to analyze a different result.

In [ ]:
# Optional: absolute path, or a path relative to the TraceML repository.
RESULT_DIR = None
PAIR = 1
SAVE_PLOT = True

In [ ]:
import json
import math
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_repo_root(start):
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "pyproject.toml").is_file() and (
            path / "examples"
        ).is_dir():
            return path
    raise FileNotFoundError("Run Jupyter from inside the TraceML repository.")


def find_result(repo_root, configured, pair):
    if configured:
        path = Path(configured).expanduser()
        return (path if path.is_absolute() else repo_root / path).resolve()

    bases = [
        repo_root / "logs/lerobot_v3_image_regression/runs",
        repo_root / "remote_logs",
    ]
    candidates = []
    for base in bases:
        if not base.is_dir():
            continue
        for path in base.iterdir():
            compare = path / "compare" / f"pair_{pair}_broken_vs_fixed.json"
            databases = [
                list(
                    (path / "traceml").glob(
                        f"pair_{pair}_{label}_*/aggregator/telemetry"
                    )
                )
                for label in ("broken", "fixed")
            ]
            if compare.is_file() and all(
                len(matches) == 1 for matches in databases
            ):
                candidates.append(path)
    if not candidates:
        raise FileNotFoundError(
            "No complete result found under logs/.../runs or remote_logs. "
            "Set RESULT_DIR to the copied experiment directory."
        )
    return max(candidates, key=lambda path: path.stat().st_mtime)


repo_root = find_repo_root(Path.cwd())
result_dir = find_result(repo_root, RESULT_DIR, PAIR)
compare_file = result_dir / "compare" / f"pair_{PAIR}_broken_vs_fixed.json"

run_dirs = {}
for label in ("broken", "fixed"):
    matches = sorted((result_dir / "traceml").glob(f"pair_{PAIR}_{label}_*"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one {label} run for pair {PAIR}, found {len(matches)}."
        )
    run_dirs[label] = matches[0]

display(Markdown(f"**Result:** `{result_dir}`"))

In [ ]:
EVENTS = {
    "input_wait_ms": ("_traceml_internal:dataloader_next", "cpu_ms"),
    "traced_step_ms": ("_traceml_internal:step_time", "gpu_ms"),
    "forward_ms": ("_traceml_internal:forward_time", "gpu_ms"),
    "backward_ms": ("_traceml_internal:backward_time", "gpu_ms"),
    "optimizer_ms": ("_traceml_internal:optimizer_step", "gpu_ms"),
}


def event_duration(events, event_name, preferred_clock):
    devices = events.get(event_name)
    if not devices:
        return None
    durations = []
    for sample in devices.values():
        value = sample.get(preferred_clock)
        if value is None:
            value = sample.get("duration_ms")
        if value is not None:
            durations.append(float(value))
    return sum(durations) if durations else None


def load_steps(database):
    uri = database.resolve().as_uri() + "?mode=ro"
    with sqlite3.connect(uri, uri=True) as connection:
        integrity = connection.execute("PRAGMA integrity_check").fetchone()[0]
        if integrity != "ok":
            raise RuntimeError(
                f"SQLite integrity check failed for {database}: {integrity}"
            )
        rows = connection.execute(
            "SELECT step, events_json FROM step_time_samples ORDER BY step"
        ).fetchall()

    samples = []
    for step, raw_events in rows:
        events = json.loads(raw_events)
        sample = {"step": int(step)}
        for metric, (event_name, clock) in EVENTS.items():
            sample[metric] = event_duration(events, event_name, clock)
        if (
            sample["input_wait_ms"] is not None
            and sample["traced_step_ms"] is not None
        ):
            sample["total_step_ms"] = (
                sample["input_wait_ms"] + sample["traced_step_ms"]
            )
        else:
            sample["total_step_ms"] = None
        samples.append(sample)
    return samples


steps = {
    label: load_steps(run_dir / "aggregator" / "telemetry")
    for label, run_dir in run_dirs.items()
}

coverage_lines = [
    "| Run | Steps | Input | Forward | Backward | Optimizer | Step time |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for label, samples in steps.items():
    counts = {
        metric: sum(sample[metric] is not None for sample in samples)
        for metric in EVENTS
    }
    total = len(samples)
    coverage_lines.append(
        f"| {label.title()} | {total} | {counts['input_wait_ms']}/{total} | "
        f"{counts['forward_ms']}/{total} | {counts['backward_ms']}/{total} | "
        f"{counts['optimizer_ms']}/{total} | {counts['traced_step_ms']}/{total} |"
    )
    if total == 0 or any(count != total for count in counts.values()):
        raise RuntimeError(f"{label} does not have complete timing coverage.")
display(Markdown("## Timing coverage\n\n" + "\n".join(coverage_lines)))

In [ ]:
comparison = json.loads(compare_file.read_text())
verdict = comparison["verdict"]
diagnosis = comparison["overview"]["primary_diagnosis"]
metrics = comparison["sections"]["step_time"]["metrics"]

rows = ["| Metric | Broken | Fixed | Change |", "|---|---:|---:|---:|"]
for name in (
    "step_time_ms",
    "input_ms",
    "forward_ms",
    "backward_ms",
    "optimizer_ms",
):
    metric = metrics[name]
    rows.append(
        f"| {metric['label']} | {metric['lhs']:.1f} ms | {metric['rhs']:.1f} ms | "
        f"{metric['pct_change']:+.1f}% |"
    )

display(
    Markdown(
        f"## TraceML comparison: {verdict['status']}\n\n"
        f"**Diagnosis:** {diagnosis['lhs']} → {diagnosis['rhs']}  \n"
        f"**Why:** {verdict['why']}\n\n" + "\n".join(rows)
    )
)

In [ ]:
def percentile(values, fraction):
    ordered = sorted(values)
    position = (len(ordered) - 1) * fraction
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight


summary_lines = [
    "| Run | Metric | Mean | Median | p95 |",
    "|---|---|---:|---:|---:|",
]
for label, samples in steps.items():
    for metric, title in (
        ("input_wait_ms", "Input wait"),
        ("total_step_ms", "Total step"),
    ):
        values = [sample[metric] for sample in samples]
        mean = sum(values) / len(values)
        median = percentile(values, 0.5)
        p95 = percentile(values, 0.95)
        summary_lines.append(
            f"| {label.title()} | {title} | {mean:.1f} ms | {median:.1f} ms | {p95:.1f} ms |"
        )
display(Markdown("## Distribution summary\n\n" + "\n".join(summary_lines)))

In [ ]:
colors = {"broken": "#D1495B", "fixed": "#00798C"}
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for label, samples in steps.items():
    x = [sample["step"] for sample in samples]
    axes[0].plot(
        x,
        [sample["input_wait_ms"] for sample in samples],
        label=label.title(),
        color=colors[label],
        linewidth=1.5,
    )
    axes[1].plot(
        x,
        [sample["total_step_ms"] for sample in samples],
        label=label.title(),
        color=colors[label],
        linewidth=1.5,
    )

axes[0].set_title(
    "DataLoader starvation appears as repeated input-wait spikes"
)
axes[0].set_ylabel("Input wait (ms)")
axes[1].set_title("The same stalls dominate end-to-end step time")
axes[1].set_ylabel("Total step time (ms)")
axes[1].set_xlabel("Training step")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)
fig.suptitle(f"LeRobot v3 image loading: pair {PAIR}", fontsize=14)
fig.tight_layout()

plot_file = result_dir / f"pair_{PAIR}_step_timing.png"
if SAVE_PLOT:
    fig.savefig(plot_file, dpi=180, bbox_inches="tight")
    print(f"Saved {plot_file}")
plt.show()

## Reading the plot

Repeated high input-wait points show the training loop exhausting prefetched batches and waiting for DataLoader workers. They demonstrate starvation, not its implementation-level cause. The controlled broken/fixed revisions connect that starvation to the upstream dataset-access change; stable forward, backward, and optimizer times help rule out model compute as the source of the improvement.